In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

In [ ]:
colors = ["#5DADE2", "white", "#FF69B4"]  # Lighter sky blue and hot pink
custom_cmap = LinearSegmentedColormap.from_list("light_skyblue_to_hot_pink", colors)

## Load data

In [ ]:
adata_full = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

fig_dir = "Reproducibility/Results/Plots/B/"
os.makedirs(fig_dir, exist_ok = True)

In [ ]:
tmp_subset = 'B'

adata = adata_full[adata_full.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["B_naive","B_memory",'Atypical_B',"GC_B","Plasma"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

## Visualization

### UMAP

In [ ]:
# Fig.S6A
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    frameon=False,
    #legend_loc="on data",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}FigureS6A_UMAP.pdf", bbox_inches='tight')
plt.close()

### RNA dotplot

In [ ]:
adata_RNA = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

adata_RNA.layers['scaled'] = sc.pp.scale(adata_RNA, copy=True).X

In [ ]:
marker_genes = ["BACH2","FOXP1","FCRL1",'FCER2',
                "BANK1",'BCL2','SSPN',
                "ZEB2","TOX","EBF1",
                'IL4R','CD22',"LRMP","MME",'RGS13',
                "JCHAIN","MZB1",'XBP1','PRDM1','CD38','SLAMF7'
                ]

In [ ]:
# Fig.S6A
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby = "celltype",
              dendrogram=False,
              cmap=custom_cmap,
              dot_max = 0.8,
              layer = 'scaled',
              use_raw=False,
              vmin = -1,
              vmax = 1,
              show = False)
plt.savefig(f"{fig_dir}FigureS6A_RNA_dotplot.pdf", bbox_inches='tight')
plt.close()

### Protein heatmap

In [ ]:
from muon import prot as pt
adt_df = adata_RNA.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_RNA.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins = ['CD22','IgM','IgD','CD11c','FcRL4','CD21','CD10','CD38','CD27']

In [ ]:
# Fig.S6A
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}FigureS6A_protein_heatmap.pdf", bbox_inches='tight')
plt.close()

### Pseudotime UMAP

In [ ]:
B_meta = pd.read_csv("Reproducibility/Results/CellRank2/B/Atypical_B_dpt_pseudotime.txt", sep='\t', index_col=0)

adata.obs['dpt_pseudotime'] = adata.obs.index.map(B_meta['dpt_pseudotime'])

In [ ]:
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)

cmap = plt.get_cmap("gnuplot").copy()
cmap.set_bad(color="#F2F2F2")

# Plot with modified colormap
sc.pl.embedding(
    adata,
    basis="umap",
    color=["dpt_pseudotime"],
    color_map=cmap,
    legend_loc="on data",
    show=False
)

plt.savefig(f"{fig_dir}FigureS6D_UMAP_pseudotime.pdf", bbox_inches='tight')
plt.close()